# Dark Data Medicine — Worked Analysis Example

This notebook demonstrates loading the registry's dataset into pandas
for filtering, cross-tabulation, and export, using nothing but the
raw JSON files under `data/` -- no API, no database required.

Every cell below was executed against the actual seed dataset shipped
in this repository (`v1.0.0`) at the time this notebook was built. If
you re-run it after the planned bulk import (see the Roadmap section
of `README.md`), the outputs will reflect the larger, real dataset.

In [1]:
import json
from pathlib import Path
import pandas as pd

KNOWN_DOMAIN_DIRS = [
    "oncology", "neurology", "pharmacology", "cardiology",
    "psychiatry", "immunology", "infectious_disease", "endocrinology",
]

def load_entries(data_dir="data", domain_filter=None):
    rows = []
    domain_dirs = [domain_filter] if domain_filter else KNOWN_DOMAIN_DIRS
    for dd in domain_dirs:
        d = Path(data_dir) / dd
        if not d.is_dir():
            continue
        for f in sorted(d.glob("*.json")):
            entry = json.load(open(f, encoding="utf-8"))
            flat = dict(entry)
            iv = flat.pop("tested_intervention", {}) or {}
            flat["intervention_type"] = iv.get("type")
            flat["intervention_name"] = iv.get("name")
            rows.append(flat)
    return pd.DataFrame(rows)

df = load_entries("data")
print(f"Loaded {len(df)} entries")
df.head()

Loaded 7 entries


,experiment_id,domain,target_disease,hypothesis,outcome,methodology_summary,researcher_orcid,institution_type,date_concluded,source_type,source_url,license,contact_email_optional,keywords,intervention_type,intervention_name
0,ONC_2025_0001,Oncology,Non-Small Cell Lung Cancer (NSCLC),Example_Compound_A improves progression-free s...,Trial terminated early; primary endpoint (prog...,"Randomized controlled trial, phase II. See reg...",0000-0000-0000-0000,University Research Lab,2024-06-01,public_database_extraction,https://clinicaltrials.gov/,CC0-1.0,,"[negative result, oncology, phase II, terminat...",Drug,Example_Compound_A
1,NEU_2025_0001,Neurology,Early-stage Alzheimer's Disease,Example_Compound_B slows cognitive decline ver...,No statistically significant difference in cog...,"Randomized, double-blind, placebo-controlled t...",0000-0000-0000-0000,University Research Lab,2024-09-12,public_database_extraction,https://clinicaltrials.gov/,CC0-1.0,,"[negative result, neurology, phase III, illust...",Drug,Example_Compound_B (illustrative)
2,PHARM_2025_0001,Pharmacology,Type 2 Diabetes Mellitus,Example_Compound_C improves insulin sensitivit...,No significant improvement in HOMA-IR index co...,"Preclinical in vivo study, diet-induced obesit...",0000-0000-0000-0000,Pharmaceutical Company,2025-02-20,original_submission,,CC-BY-4.0,,"[negative result, pharmacology, preclinical, i...",Molecule,Example_Compound_C (illustrative)
3,CARD_2025_0001,Cardiology,Heart Failure with Preserved Ejection Fraction...,Example_Compound_D reduces hospitalization for...,Trial stopped early for futility at interim an...,"Multi-center randomized controlled trial, phas...",0000-0000-0000-0000,Hospital / Clinical Center,2024-11-30,public_database_extraction,https://clinicaltrials.gov/,CC0-1.0,,"[negative result, cardiology, futility stop, i...",Drug,Example_Compound_D (illustrative)
4,PSYCH_2025_0001,Psychiatry,Treatment-Resistant Major Depressive Disorder,Example_Digital_Intervention_E reduces MADRS d...,No significant between-group difference in MAD...,"Randomized controlled trial, waitlist control,...",0000-0000-0000-0000,University Research Lab,2025-01-18,literature_mining,,CC-BY-4.0,,"[negative result, psychiatry, digital therapeu...",Behavioral,Example_Digital_Intervention_E (illustrative)


## Entries per domain

In [2]:
df["domain"].value_counts()

domain
Oncology              1
Neurology             1
Pharmacology          1
Cardiology            1
Psychiatry            1
Immunology            1
Infectious Disease    1
Name: count, dtype: int64

## Entries per intervention type

In [3]:
df["intervention_type"].value_counts()

intervention_type
Drug          4
Molecule      1
Behavioral    1
Biologic      1
Name: count, dtype: int64

## Filtering: institution types represented

Useful, for instance, for a funder (see `MANIFEST.md` §I / white paper
§22.4) checking coverage across academic vs. industry vs. government
sources before drawing conclusions from a subset.

In [4]:
df.groupby("institution_type")["experiment_id"].apply(list)

institution_type
Government Institute                                           [ID_2025_0001]
Hospital / Clinical Center                                   [CARD_2025_0001]
Pharmaceutical Company                       [PHARM_2025_0001, IMM_2025_0001]
University Research Lab       [ONC_2025_0001, NEU_2025_0001, PSYCH_2025_0001]
Name: experiment_id, dtype: object

## Exporting for use outside Python

Both lines below are commented out by default so running this
notebook end-to-end doesn't write files as a side effect -- uncomment
whichever you need.

In [5]:
# df.to_csv("darkdata_flat.csv", index=False)
# df.to_parquet("darkdata_flat.parquet", index=False)
print("Uncomment the lines above to export.")

Uncomment the lines above to export.


## Note on evidentiary status

Every entry loaded above is, as of this release, an illustrative seed
example (see each entry's `keywords` field) rather than a curator-
reviewed real-world submission. Treat any analysis run against the
current dataset as a demonstration of the pipeline, not as a finding.
See `docs/FAQ.md` and `MANIFEST.md` §III for the project's own
statement of its current evidentiary limits.